<a href="https://colab.research.google.com/github/ProfAndersonVanin/IBM3130-PLN-2026/blob/main/semana-04/Aula04_BoW_TFIDF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📘 IBM3130 — Processamento de Linguagem Natural
## Aula 04 · Bag of Words e TF-IDF

---

Nesta aula vamos aprender a **transformar textos em números**.  
Isso é necessário porque algoritmos de Machine Learning só trabalham com números.

Você verá dois métodos:
- **Bag of Words (BoW):** conta quantas vezes cada palavra aparece
- **TF-IDF:** mede a importância de cada palavra no documento

---


---
## ⚙️ Passo 0 — Preparar o ambiente

Execute esta célula **uma única vez** antes de começar.

> 💡 O scikit-learn já vem instalado no Colab — não precisa de `pip install`.

In [ ]:
# Importar as bibliotecas que vamos usar
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
import numpy as np
import pandas as pd


---
## 💡 Por que transformar texto em números?

Algoritmos de Machine Learning não leem palavras — eles leem **números**.  
Então precisamos converter cada texto em um vetor (lista) de números.

A ideia mais simples: **contar quantas vezes cada palavra aparece** no texto.

Isso é o **Bag of Words** — literalmente uma 'sacola de palavras'.

In [ ]:
reviews = ["A bateria dura o dia todo sem precisar recarregar.",
           "O som é muito limpo e os graves são fortes.",
           "A câmera tira fotos lindas mesmo em lugares escuros.",
           "O sistema abre os aplicativos muito rápido e sem travar.",
           "A tela tem cores vivas e imagens bem nítidas.",
           "O celular esquenta muito quando uso jogos pesados.",
           "O plástico da carcaça arranha com muita facilidade.",
           "O carregador que vem na caixa é muito lento."]

In [ ]:
teste = ["para para para para para para gato gato marrom canta",
         "para para para para corvo preto gato pia"]

---
# 📌 EXEMPLO 1 — Bag of Words

Vamos usar o `CountVectorizer` do scikit-learn para fazer o BoW automaticamente.

Ele faz três coisas:
1. Descobre o vocabulário de todos os textos
2. Conta quantas vezes cada palavra aparece em cada texto
3. Devolve uma tabela com esses números

In [ ]:
# Criar o CountVectorizer
cv = CountVectorizer()

# Aplicar nos textos — fit_transform faz tudo automaticamente
matriz_bow_teste = cv.fit_transform(teste)

# Ver o vocabulário descoberto
vocabulario_teste = cv.get_feature_names_out()
print('Vocabulário descoberto:')
print(list(vocabulario_teste))
print()
print('Total de palavras únicas:', len(vocabulario_teste))

Representação das palavras e suas colunas

In [ ]:
print(cv.vocabulary_)

In [ ]:
matriz_bow_teste

Exibindo a matriz

In [ ]:
bow_teste = matriz_bow_teste.toarray()
bow_teste

---
# 📌 EXEMPLO 2 — TF-IDF: Frequência × Importância

O **TF-IDF** resolve o problema do BoW usando uma lógica simples:

- **TF** (Term Frequency): quantas vezes a palavra aparece **neste texto**
- **IDF** (Inverse Document Frequency): quão **rara** a palavra é no corpus inteiro

O resultado é:
- Palavras **frequentes neste texto E raras no corpus** → peso **alto** ✅
- Palavras **frequentes em todos os textos** → peso **baixo** (próximo de zero) ❌

```
TF-IDF = TF × IDF
```

### Etapa 1 — Aplicar o TF-IDF nos mesmos textos

In [ ]:
# TfidfVectorizer funciona igual ao CountVectorizer
# mas calcula pesos em vez de contagens simples
tfidf = TfidfVectorizer()
#tfidf = TfidfVectorizer(norm=None)

tfidf_transform = tfidf.fit_transform(teste)

In [ ]:
vocabulario_tfidf = tfidf.get_feature_names_out()
print(vocabulario_tfidf)

In [ ]:
print(tfidf.vocabulary_)

In [ ]:
tfidf_words = tfidf_transform.toarray()
tfidf_words

---
# 📌 EXEMPLO — Classificador de Sentimento

Agora vamos usar o TF-IDF para **classificar** reviews de produtos como positivas ou negativas.

O classificador que vamos usar é o **Naive Bayes** — um dos mais usados em classificação de texto.

O pipeline tem 3 etapas:
```
texto → TF-IDF → Naive Bayes → classe (positivo / negativo)
```

### Etapa 1 — Os dados: reviews e rótulos

Vamos criar um conjunto de reviews com seus rótulos:
- `'pos'` = review positiva
- `'neg'` = review negativa

In [ ]:
# Reviews de produtos
reviews = [
    'produto excelente qualidade otima recomendo',
    'chegou rapido funciona perfeitamente adorei',
    'material muito bom vale o preco',
    'produto bonito e resistente recomendo',
    'pessimo produto parou de funcionar lixo',
    'horrivel chegou quebrado nao recomendo',
    'qualidade ruim fragil decepcionante',
    'produto defeituoso suporte pessimo',
]

# Rótulos: pos = positivo, neg = negativo
rotulos = ['pos', 'pos', 'pos', 'pos',
           'neg', 'neg', 'neg', 'neg']

print('Reviews e rótulos:')
for review, rotulo in zip(reviews, rotulos):
    emoji = '✅' if rotulo == 'pos' else '❌'
    print(f'  {emoji} [{rotulo}] "{review}"')

---
### Etapa 2 — Dividir em treino e teste

Não podemos testar o modelo com os mesmos dados que usamos para treiná-lo.  
É como dar a prova antes e testar com as mesmas questões — não mede o aprendizado real.

Vamos separar manualmente: **6 reviews para treino** e **2 para teste**.

In [ ]:
# Treino: primeiras 3 positivas e primeiras 3 negativas
reviews_treino = reviews[:3] + reviews[4:7]
rotulos_treino = rotulos[:3] + rotulos[4:7]

# Teste: última positiva e última negativa
reviews_teste  = [reviews[3], reviews[7]]
rotulos_teste  = [rotulos[3], rotulos[7]]

print('TREINO (' + str(len(reviews_treino)) + ' reviews):')
for r, l in zip(reviews_treino, rotulos_treino):
    emoji = '✅' if l == 'pos' else '❌'
    print(f'  {emoji} "{r}"')

print()
print('TESTE (' + str(len(reviews_teste)) + ' reviews — o modelo NÃO viu essas):')
for r, l in zip(reviews_teste, rotulos_teste):
    emoji = '✅' if l == 'pos' else '❌'
    print(f'  {emoji} "{r}"')

---
### Etapa 3 — Treinar o classificador

Vamos:
1. Transformar os textos de treino em vetores TF-IDF
2. Treinar o Naive Bayes com esses vetores

In [ ]:
# Passo 1: criar e treinar o TF-IDF com os dados de TREINO
tfidf_clf = TfidfVectorizer()
X_treino  = tfidf_clf.fit_transform(reviews_treino)

print('TF-IDF treinado!')
print('Vocabulário do treino:', list(tfidf_clf.get_feature_names_out()))
print()

# Passo 2: treinar o Naive Bayes
modelo = MultinomialNB()
modelo.fit(X_treino, rotulos_treino)

print('Modelo Naive Bayes treinado!')
print('Classes aprendidas:', modelo.classes_)

---
### Etapa 4 — Testar o classificador

Agora testamos com as **2 reviews que o modelo nunca viu**.

In [ ]:
# Passo 1: transformar os textos de TESTE com o TF-IDF já treinado
# Usamos transform() — não fit_transform()!
# (o vocabulário já foi aprendido no treino)
X_teste = tfidf_clf.transform(reviews_teste)

# Passo 2: fazer as predições
predicoes = modelo.predict(X_teste)

print('=== Resultados no teste ===')
print()

for review, real, pred in zip(reviews_teste, rotulos_teste, predicoes):
    emoji_real = '✅' if real == 'pos' else '❌'
    emoji_pred = '✅' if pred == 'pos' else '❌'
    correto    = '✓ CERTO' if real == pred else '✗ ERRADO'

    print(f'Review: "{review}"')
    print(f'  Real:      {emoji_real} {real}')
    print(f'  Predição:  {emoji_pred} {pred}  {correto}')
    print()

---
### Etapa 5 — Testar com reviews novas

O modelo está treinado. Agora podemos usá-lo para classificar qualquer nova review!

In [ ]:
# Reviews completamente novas — o modelo nunca viu
novas = [
    'produto incrivel chegou rapido muito bom',
    'nao funciona pessimo produto horrivel',
    'vale o preco qualidade boa recomendo',
]

# Transformar com o TF-IDF já treinado
X_novas = tfidf_clf.transform(novas)

# Classificar
pred_novas = modelo.predict(X_novas)

print('=== Classificando novas reviews ===')
print()

for review, pred in zip(novas, pred_novas):
    emoji  = '✅' if pred == 'pos' else '❌'
    classe = 'POSITIVO' if pred == 'pos' else 'NEGATIVO'
    print(f'{emoji} {classe}')
    print(f'   "{review}"')
    print()

---
### 📝 Reflexão — Exemplo 3

**Pergunta 1:** Por que usamos `transform()` nos dados de teste e não `fit_transform()`?

**Pergunta 2:** O modelo acertou as predições nos dados de teste? Por que pode errar?

**Pergunta 3:** Se você adicionar a review `'produto bom mas chegou atrasado'`, o modelo classificaria como positivo ou negativo? Por quê?

*(Escreva sua resposta aqui)*

---
# ✏️ Desafio — Use seus próprios textos!

Substitua os textos abaixo e veja o classificador funcionando com o seu próprio exemplo.

> Pode ser avaliações de um restaurante, de um filme, de um aplicativo...  
> Qualquer coisa que tenha texto positivo e negativo!

In [ ]:
# ╔══════════════════════════════════════════════╗
# ║  SUBSTITUA OS TEXTOS AQUI!                  ║
# ╚══════════════════════════════════════════════╝

meus_textos = [
    'coloque aqui um texto positivo',
    'coloque aqui outro texto positivo',
    'coloque aqui um texto negativo',
    'coloque aqui outro texto negativo',
]

meus_rotulos = ['pos', 'pos', 'neg', 'neg']

# ── Pipeline completo ──────────────────────────

# Passo 1: TF-IDF
meu_tfidf   = TfidfVectorizer()
meu_X       = meu_tfidf.fit_transform(meus_textos)

# Passo 2: treinar o modelo
meu_modelo  = MultinomialNB()
meu_modelo.fit(meu_X, meus_rotulos)

# Passo 3: testar com um novo texto
novo_texto  = ['coloque aqui um texto para classificar']
X_novo      = meu_tfidf.transform(novo_texto)
pred_novo   = meu_modelo.predict(X_novo)

# Mostrar resultado
print('Vocabulário aprendido:')
print(list(meu_tfidf.get_feature_names_out()))
print()
print('Texto para classificar:')
print(' ', novo_texto[0])
print()

emoji  = '✅' if pred_novo[0] == 'pos' else '❌'
classe = 'POSITIVO' if pred_novo[0] == 'pos' else 'NEGATIVO'
print(f'Classificado como: {emoji} {classe}')

---
## ✅ O que aprendemos nesta aula

| Conceito | O que faz |
|----------|-----------|
| **Bag of Words** | Conta quantas vezes cada palavra aparece |
| **Esparsidade** | A maioria dos valores é zero — é normal! |
| **TF-IDF** | Dá peso maior para palavras específicas e peso menor para palavras comuns |
| **CountVectorizer** | Implementa o BoW no scikit-learn |
| **TfidfVectorizer** | Implementa o TF-IDF no scikit-learn |
| **Naive Bayes** | Classificador que aprende com os vetores TF-IDF |
| **fit_transform** | Aprende o vocabulário E transforma (use só no treino!) |
| **transform** | Só transforma — sem aprender (use no teste e nas novas reviews) |

---

### 📚 Referências

- **B1** Caseli & Nunes. *Processamento de linguagem natural*. BPLN, 2023. Cap. 6, pp. 149–180.
- **B2** Faceli et al. *Inteligência Artificial*. LTC, 2025. Cap. 5 e 8.
- **C3** Sicsú et al. *Técnicas de Machine Learning*. Blucher, 2023. Cap. 4.

---

### ⏭️ Próxima aula — Semana 05

Na **Aula 05** vamos aprender sobre **modelos N-gram**: bigramas, trigramas  
e como calcular a probabilidade de uma sequência de palavras.

---

*IBM3130 · PLN · Aula 04 · 2º Semestre de 2026*